---
title: Alignment Format Validation
subtitle: SAM/BAM file format checks
date: "9999-12-31"
edit_url: null
---
**Linked-read type**: PLACEHOLDER

In [ ]:
from harpy.report.components import StatsBox, standard_itable
import itables
import pandas as pd
itables.init_notebook_mode()


This report reflects the BAM files Harpy processed to identify obvious formatting issues that may require your attention.

In [ ]:
infile = "validate.bam.tsv"


In [ ]:
data = pd.read_table(infile, sep=r'\s+')
attention_df = data.drop(data.columns[[0, 1]], axis=1).apply(
    lambda row: (row[['nameMismatch', 'noBX', 'badBX']].sum()), axis=1
)
attention = (attention_df > 0).sum()
noMItag = (data['noMI'] == data['alignments']).sum()
noBXtag = (data['noBX'] == data['alignments']).sum()
bxnotlast = (data['bxNotLast'] > 0).sum()
format_issues = (data['badBX'] > 0).sum()


In [ ]:
(
    StatsBox()
    .add(len(data.index), "Files")
    .conditional(attention, "Needs Review", 1, lower_bad=False)
    .conditional(noBXtag, "No BX Tag", 1, lower_bad=False)
    .conditional(noMItag, "No MI Tag", 1, lower_bad=False)
    .conditional(format_issues, "Bad Format", 1, lower_bad=False)
    .conditional(bxnotlast, "BX Not Last", 1, lower_bad=False)
    .render()
)


## Metrics
The `harpy validate bam` command created a `validate.bam.tsv` file
that summarizes the results included in this report. This file contains
a tab-delimited table with the columns: `file`,	`nameMismatch`, `alignments`, `format`, `noBX`, and `badBX`.
These columns are defined in **Supporting info** below.

Look over the metric descriptions to better understand what these validation assessments mean and their severities. At the bottom of this report
is additional information regarding formatting.

:::{note} Validation logic
:open:
Severity given by: 🔶 = moderate  | 🛑 = serious
| column            | severity  |  pass condition                                                              |  fail condition 🟨                                       |
|:------------------|:----|:---------------------------------------------------------------------------------|:-----------------------------------------------------------|
| **nameMismatch** |  🛑 | the file name matches the `@RG ID:` tag in the header                            | file name does not match `@RG ID:` in the header           |
| **noMI**        |  🔶/🛑  | **any** alignments with `BX:Z` tag also have `MI`                                | **all** reads have `BX:Z` tag present but `MI` tag absent  |
| **noBX**      |  🛑  | **any** `BX:Z` tags present                                                      | **all** alignments lack `BX:Z` tag                         |
| **bxNotLast**     |  🔶  | **all** reads have `BX:Z` as final tag in alignment records                      | **at least 1 read** doesn't have `BX:Z` tag as final tag   |
| **badBX**        |  🛑  | **all** alignments with `BX:Z` tag have properly formatted barcodes for their chemistry | **any** `BX:Z` barcodes have incorrectly formatted barcodes for their chemistry |

:::


In [ ]:
standard_itable(
    data,
    "validate.bam",
    fixedcols = 1,
    caption="Alignment file format validation. Validation failures will have their cells highlighted in yellow.",
    coldefs=[
        {
            "targets": [3,4,5,6],
            "createdCell": itables.JavascriptFunction(
                """
function (td, cellData, rowData, row, col) {
    if (cellData > 0) {
        $(td).css('background-color', '#f6ab3c')
        $(td).css('color', 'black')
    }
}
"""
            )
        }
    ]
)


## Supporting info
:::{dropdown} Understanding the columns
:open:
**file**: The name of the BAM file

**nameMismatch**: The sample name of the file inferred from name of the file (i.e. Harpy assumes `sample1.bam` to be `sample1`) does not match the `@RG ID` tag in the alignment file.

**alignments**: The total number of alignment records in the file

**noMI**: Alignment records lack `MI` tag (`MI:i` or `MI:Z`)

**noBX**: The number of alignments that do not have a `BX:Z` tag in the record.
If you expect all or some of your reads should have `BX:Z` tags, then further investigation is necessary

**badBX**: The barcode in the `BX:Z` tag does not adhere to the proper chemistry-specific format

**bxNotLast**: The `BX:Z` tag in the alignment record is not the last tag.
Only relevant for LEVIATHAN variant calling and can be ignored if not intending to call
structural variants with LEVIATHAN,otherwise the `BX:Z` tag must be the last comment in an alignment
record. `harpy align` will automatically move the `BX:Z` tag to the end of the alignment record.
:::


:::{dropdown} What is "valid"?
A "valid" linked-read alignment file will contain a `BX:Z` tag with an alignment record
that features a properly-formatted barcode. Note that the `invalidated` definitions provided
below refer to how each chemistry defines an "invalid barcode", meaning it adheres to the chemistry's
format, but specifies it is not reliably linked to any other molecule. The formats are:

**haplotagging**: `AXXCXXBXXDXX` where `X` is a 0-9 integer
- example: `A02C56B09D88`
- invalidated: if any position is `00` (e.g. `A91C00B42D57`)

**stlfr**: `X_X_X` where `X` is any integer
- example: `9_851_22 
- invalidated: if any position is `0` (e.g. `51_0_492`)

**tellseq/10x**: `ATCGN` nucleotide letters
- example: `ATATTTACGGGAC`
- invalidated: if any nucleotide is `N` (e.g. `ATACANGGAT`)

If a barcode is not in the technology-specific format, then it's likely the input FASTQ used for read mapping is the source of the
issue. You can check those FASTQ files for errors with `harpy validate fastq`. 
:::

**Example of a valid record**

Below is an example of a proper alignment record for a file named `sample1.bam`.
Note the tag `RG:Z:sample1`, indicating this alignment is associated with `sample1` and
matches the file name. Also note the correctly formatted haplotagging barcode `BX:Z:A19C01B86D78`
and the presence of a `MI:` tag. To reduce horizontal scrolling, the alignment sequence and Phred
scores have been replaced with `SEQUENCE` and `QUALITY`, respectively.

```
A00814:267:HTMH3DRXX:2:1132:26268:10316 113     contig1 6312    60      4S47M1D86M      =       6312    0       SEQUENCE  QUALITY     NM:i:2  MD:Z:23C23^C86  MC:Z:4S47M1D86M AS:i:121        XS:i:86 RG:Z:sample1   MI:i:4040669   BX:Z:A19C01B86D78
```

:::